# 05 가상 유전자 삭제의 해석

**실측 섭동 검증 전 탐색.** 무엇을 하는가: train hepatocyte 최대8세포에서 ALB/HNF4A를 각각 독립 삭제하고 no-op 및 발현량 유사 random gene 대조와 비교한다. 치료 효과나 발현 예측 실습이 아니다.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/paths.yaml').exists()
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', MKL_NUM_THREADS='8', TOKENIZERS_PARALLELISM='false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import yaml, pandas as pd, numpy as np
PATHS = yaml.safe_load((ROOT / 'configs/paths.yaml').read_text())
DATA = Path(PATHS['data_root'])
def run_script(name, config=None, model='scgpt', args=()):
    cmd = [str(ROOT / f'.venvs/vcell-{model}/bin/python'), '-u', str(ROOT / 'scripts' / name)]
    if config: cmd += ['--config', str(ROOT / 'configs' / config)]
    cmd += list(args)
    print(' '.join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout[-16000:])
    if p.returncode:
        print(p.stderr[-16000:])
        p.check_returncode()
    return p
print('code:', ROOT, '\ndata:', DATA)


code: /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver 
data: /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver


In [2]:
run_script('06_geneformer_virtual_perturb.py', 'tutorial_isp_smoke.yaml', model='geneformer')

/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-geneformer/bin/python -u /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/06_geneformer_virtual_perturb.py --config /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_isp_smoke.yaml


                                                      count          mean           max
gene  condition                        token_present                                   
ABCD1 expression_matched_random_delete False              6  9.934107e-09  5.960464e-08
                                       True               2  1.411438e-04  2.796650e-04
ALB   independent_delete               False              2  2.980232e-08  5.960464e-08
                                       True               6  1.409948e-04  3.375411e-04
FTL   expression_matched_random_delete True               8  2.869293e-04  5.840063e-04
HNF4A independent_delete               False              6  9.934107e-09  5.960464e-08
                                       True               2  2.622604e-06  2.622604e-06
none  no_op                            True               8  1.490116e-08  5.960464e-08



CompletedProcess(args=['/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-geneformer/bin/python', '-u', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/06_geneformer_virtual_perturb.py', '--config', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_isp_smoke.yaml'], returncode=0, stdout='                                                      count          mean           max\ngene  condition                        token_present                                   \nABCD1 expression_matched_random_delete False              6  9.934107e-09  5.960464e-08\n                                       True               2  1.411438e-04  2.796650e-04\nALB   independent_delete               False              2  2.980232e-08  5.960464e-08\n                                       True               6  1.409948e-04  3.375411e-04\nFTL   expression_matched_random_delete True               8  2.869293e-04  5.840063e-04\nHNF4A independent_delete    

In [3]:
table = pd.read_csv(ROOT / 'reports/virtual_perturbation_embedding_shift_smoke.tsv', sep='\t')
print(table.groupby(['gene','condition','token_present']).cosine_shift.agg(['count','mean','median','max']).to_string())
assert (table.loc[table.condition == 'no_op','cosine_shift'] < 1e-5).all()
print('no-op 확인 통과; token_present=False는 실제 삭제된 세포가 아님')

                                                      count          mean        median           max
gene  condition                        token_present                                                 
ABCD1 expression_matched_random_delete False              6  9.934107e-09  0.000000e+00  5.960464e-08
                                       True               2  1.411438e-04  1.411438e-04  2.796650e-04
ALB   independent_delete               False              2  2.980232e-08  2.980232e-08  5.960464e-08
                                       True               6  1.409948e-04  1.259148e-04  3.375411e-04
FTL   expression_matched_random_delete True               8  2.869293e-04  2.326667e-04  5.840063e-04
HNF4A independent_delete               False              6  9.934107e-09  0.000000e+00  5.960464e-08
                                       True               2  2.622604e-06  2.622604e-06  2.622604e-06
none  no_op                            True               8  1.490116e-08  0.00000

생물학적 해석: 출력은 rank token 삭제 전후 CLS embedding의 1−cosine이다. 큰 변화가 강한 생물학적 영향·약효·생존율을 뜻하지 않는다. post-perturbation expression이나 logFC로 바꿔 해석하지 않는다. 신뢰할 목표 상태가 없어 변화의 크기만 본다. 분류 confidence를 정상화로 읽지 않는다.

실패 해결: 실제 발현 및 vocabulary를 확인해야 한다. 여러 genes를 함께 넣는 공동 삭제 대신 한 gene씩 처리한다. token이 없는 세포는 별도 표시한다. 실제 KO/CRISPR 반응 예측에는 대조군·실측 개입 데이터와 별도 검증이 필요하다.